In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# -------------------- CONFIGURATION --------------------
DATA_DIR = None
IMG_SIZE = (48, 48)
BATCH_SIZE = 64
EPOCHS = 30
SEED = 42
keras.utils.set_random_seed(SEED)

In [ ]:
# -------------------- GPU / CPU CHECK --------------------
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU available:", gpus[0].name)
else:
    print("GPU not found. Training using CPU.")

In [ ]:
# -------------------- DOWNLOAD DATASET --------------------
if DATA_DIR is None:
    try:
        import kagglehub
        print("Downloading FER-2013 dataset...")
        DATA_DIR = kagglehub.dataset_download("msambare/fer2013")
    except Exception as e:
        raise SystemExit(
            "Dataset download failed. Install kagglehub using:\n"
            "!pip install kagglehub\n"
            "Then check your internet connection and Kaggle access."
        ) from e
DATA_DIR = pathlib.Path(DATA_DIR)
train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"
if not train_dir.exists() or not test_dir.exists():
    raise SystemExit(
        f"Dataset folders not found in {DATA_DIR}. "
        "Expected train/ and test/ folders."
    )
print("\nDataset location:", DATA_DIR)

In [ ]:
# -------------------- LOAD DATASET --------------------
train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    validation_split=0.1,
    subset="training",
    seed=SEED
)
val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    validation_split=0.1,
    subset="validation",
    seed=SEED
)
test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)
class_names = train_ds.class_names
num_classes = len(class_names)
print("\nEmotion classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:
# -------------------- CLASS WEIGHTS --------------------
y_train = np.concatenate([y.numpy() for _, y in train_ds])
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weights = dict(enumerate(weights))
print("\nClass weights:")
for i, name in enumerate(class_names):
    print(name, ":", round(float(class_weights[i]), 2))

In [ ]:
# -------------------- DISPLAY DATASET --------------------
counts = np.bincount(y_train, minlength=num_classes)
plt.figure(figsize=(9, 4))
plt.bar(class_names, counts)
plt.title("Training Images Per Emotion")
plt.xlabel("Emotion")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
# Display sample images
plt.figure(figsize=(12, 4))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- OPTIMIZE DATA PIPELINE --------------------
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(
    1000, seed=SEED
).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
# -------------------- BUILD CNN MODEL --------------------
model = keras.Sequential([
    layers.Input(shape=(48, 48, 1)),
    layers.Rescaling(1.0 / 255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    # First convolution block
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    # Second convolution block
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    # Third convolution block
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    # Fully connected layers
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    # Output layer
    layers.Dense(num_classes, activation="softmax")
])

In [ ]:
# -------------------- COMPILE MODEL --------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# -------------------- CALLBACKS --------------------
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_emotion_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )
]

In [ ]:
# -------------------- TRAIN MODEL --------------------
print("\nStarting CNN training...\n")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
# -------------------- EVALUATE MODEL --------------------
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print("\n========== MODEL RESULTS ==========")
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))

In [ ]:
# -------------------- ACCURACY AND LOSS GRAPHS --------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["accuracy"], label="Training Accuracy")
ax[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
ax[0].set_title("CNN Accuracy")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Accuracy")
ax[0].legend()
ax[1].plot(history.history["loss"], label="Training Loss")
ax[1].plot(history.history["val_loss"], label="Validation Loss")
ax[1].set_title("CNN Loss")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Loss")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- CLASSIFICATION REPORT --------------------
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
))

In [ ]:
# -------------------- CONFUSION MATRIX --------------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Emotion")
plt.ylabel("Actual Emotion")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- DISPLAY PREDICTIONS --------------------
plt.figure(figsize=(12, 6))
for images, labels in test_ds.take(1):
    predictions = model.predict(images[:8], verbose=0)
    for i in range(min(8, len(images))):
        predicted_class = int(np.argmax(predictions[i]))
        actual_class = int(labels[i])
        confidence = float(np.max(predictions[i])) * 100
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(
            f"Predicted: {class_names[predicted_class]}\n"
            f"Actual: {class_names[actual_class]}\n"
            f"Confidence: {confidence:.1f}%",
            fontsize=8,
            color="green" if predicted_class == actual_class else "red"
        )
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- SAVE MODEL --------------------
model.save("emotion_cnn_final.keras")
print("\nModel saved successfully!")
print("Best model: best_emotion_cnn.keras")
print("Final model: emotion_cnn_final.keras")
print("\nFacial Emotion Recognition Completed Successfully!")